# Actividad 3.1 - Valores Atípicos
## Ventas por Asesor (ventas_por_asesor_sin_nulos.csv)

Detección y tratamiento de outliers con dos métodos: Desviación Estándar y Rango Intercuartílico (IQR).

**Nota:** las variables de esta sección llevan el sufijo `_va` para no chocar con las del análisis de piso, si ambos análisis viven en el mismo notebook.

### Paso 1: Importar librerías

In [ ]:
#Importamos las librerias pandas, numpy y matplotlib respectivamente
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Paso 2: Cargar el archivo CSV

In [ ]:
#Cargar archivo csv desde equipo
from google.colab import files
files.upload()

In [ ]:
#Carga desde un archivo .csv sin indice
data_va = pd.read_csv('ventas_por_asesor_sin_nulos.csv')
data_va.head()

### Paso 3: Verificar estructura y nulos

In [ ]:
#Verificamos información del DataFrame y valores nulos
data_va.info()
valores_nulos_va = data_va.isnull().sum()
valores_nulos_va

### Paso 4: Separar columnas cuantitativas y cualitativas

In [ ]:
#Creo 2 dataframes para poder procesar los outliers
cuantitativas_va = data_va.iloc[:, 1:]
cualitativas_va = data_va.iloc[:, [0]]

### Paso 5: Diagrama de caja por cada columna

In [ ]:
#Diagrama de caja individual por cada columna (mismo enfoque que en el archivo de piso, por las escalas distintas)
cols_va = cuantitativas_va.columns
n_va = len(cols_va)
n_cols_grid_va = 5
n_rows_grid_va = -(-n_va // n_cols_grid_va)  # redondeo hacia arriba

fig, axes = plt.subplots(n_rows_grid_va, n_cols_grid_va, figsize=(20, n_rows_grid_va * 2.5))
axes = axes.flatten()

for i, col in enumerate(cols_va):
    axes[i].boxplot(cuantitativas_va[col].dropna(), vert=False)
    axes[i].set_title(col, fontsize=8)
    axes[i].tick_params(labelsize=6)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## MÉTODO 1: DESVIACIÓN ESTÁNDAR

### Paso 6: Calcular límites (media ± 3 desviaciones estándar)

In [ ]:
#Método aplicando desviación estandar. Encuentro los valores extremos
y_va = cuantitativas_va
Limite_Superior_va = y_va.mean() + 3*y_va.std()
Limite_Inferior_va = y_va.mean() - 3*y_va.std()
print("Limite superior permitido", Limite_Superior_va)
print("Limite inferior permitido", Limite_Inferior_va)

### Paso 7: Marcar outliers como nulos

In [ ]:
#Obtenemos datos y los outliers se convierten en nulos en el DataFrame
data_std_va = cuantitativas_va[(y_va <= Limite_Superior_va) & (y_va >= Limite_Inferior_va)]
data_std_va

### Paso 8: Rellenar outliers con la media

In [ ]:
#Reemplazamos valores atípicos (nulos) del dataframe con "mean"
#Realizamos una copia del dataframe
data_std_clean_va = data_std_va.copy()
data_std_clean_va = data_std_clean_va.fillna(round(data_std_va.mean(), 1))
data_std_clean_va

## MÉTODO 2: RANGO INTERCUARTÍLICO (IQR)

### Paso 9: Calcular límites (Q1 - 1.5·IQR, Q3 + 1.5·IQR)

In [ ]:
#Método aplicando Cuartiles. Encuentro cuartiles 0.25 y 0.75
percentile25_va = y_va.quantile(0.25) #Q1
percentile75_va = y_va.quantile(0.75) #Q3
iqr_va = percentile75_va - percentile25_va

Limite_Superior_iqr_va = percentile75_va + 1.5*iqr_va
Limite_Inferior_iqr_va = percentile25_va - 1.5*iqr_va
print("Limite superior permitido", Limite_Superior_iqr_va)
print("Limite inferior permitido", Limite_Inferior_iqr_va)

### Paso 10: Identificar columnas donde el IQR es 0 (excluirlas del método)

In [ ]:
#Cuando IQR = 0, cualquier valor distinto de 0 se marcaría como outlier (no tiene sentido en columnas de ventas mensuales dispersas)
#Identificamos esas columnas para excluirlas del tratamiento
cols_iqr_valida_va = iqr_va[iqr_va > 0].index
cols_iqr_invalida_va = iqr_va[iqr_va == 0].index

print("Columnas EXCLUIDAS del método IQR (rango intercuartílico = 0):")
print(list(cols_iqr_invalida_va))

### Paso 11: Marcar outliers como nulos (solo columnas válidas)
**Nota:** `data_iqr_va` SÍ debe mostrar NaN aquí — son los outliers recién marcados. Es normal, se corrige en el paso siguiente.

In [ ]:
#Aplicamos el método IQR únicamente a las columnas donde el IQR > 0
data_iqr_va = cuantitativas_va.copy()
data_iqr_va[cols_iqr_valida_va] = cuantitativas_va[cols_iqr_valida_va][
    (y_va[cols_iqr_valida_va] <= Limite_Superior_iqr_va[cols_iqr_valida_va]) &
    (y_va[cols_iqr_valida_va] >= Limite_Inferior_iqr_va[cols_iqr_valida_va])
]
data_iqr_va

In [ ]:
#Corroboramos valores nulos del dataframe
valores_nulos_va = data_iqr_va.isnull().sum()
valores_nulos_va

### Paso 12: Rellenar outliers con la mediana
**Nota:** `data_iqr_clean_va` (después de esta celda) ya NO debe tener ningún NaN.

In [ ]:
#Reemplazamos valores atípicos (nulos) del dataframe con "median"
#Realizamos una copia del dataframe
data_iqr_clean_va = data_iqr_va.copy()
data_iqr_clean_va = data_iqr_clean_va.fillna(round(data_iqr_va.median(), 1))
data_iqr_clean_va

## RESULTADOS FINALES

### Paso 13: Unir cada dataframe limpio con la columna Nombre_Vendedor

In [ ]:
# Unimos el dataframe cuantitativo limpio con el dataframe cualitativo (Nombre_Vendedor)
Datos_limpios_std_va = pd.concat([cualitativas_va, data_std_clean_va], axis=1)
Datos_limpios_iqr_va = pd.concat([cualitativas_va, data_iqr_clean_va], axis=1)

Datos_limpios_std_va

### Paso 14: Exportar y descargar los CSVs

In [ ]:
# Convertimos ambos DataFrames a CSV
Datos_limpios_std_va.to_csv("ventas_asesor_std.csv", index=False)
Datos_limpios_iqr_va.to_csv("ventas_asesor_iqr.csv", index=False)

In [ ]:
# Descargamos ambos archivos
from google.colab import files
files.download("ventas_asesor_std.csv")
files.download("ventas_asesor_iqr.csv")